# Tumor–Stroma Mask Segmentation with Pre‑Trained U-Net

**Author: Fabien Bertillot**  

---

## 1. Purpose

Generate high‑resolution binary masks (tumor vs stroma) for TMA cores using a pretrained U-Net. The UNet model was trained using the U-Net (2D) multilabel notebook from https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki#contributors
The notebook is **inference‑only**: no weight updates occur.

---

## 2. Requirements

- **Python 3.9+**  
- Packages:  
  - `tensorflow` / `keras` (U-Net)  
  - `numpy`, `pandas`, `tifffile`, `scikit‑image`, `opencv‑python`  
- **Pretrained weights:** `.h5`  
- **Input directory** of images (`.tif`)  

---

## 3. Workflow Overview

1. **Load dependencies**  
2. **Check GPU access**  
3. **Model Loading** 
4. **Post‑processing and outputs**  
   - Binary mask (`*_mask.png`)  
---


## 1) Load Dependencies

In [5]:
from PIL import Image
import numpy as np
from skimage.util.shape import view_as_windows
from tqdm.notebook import tqdm
from skimage import img_as_ubyte, io, transform
import tensorflow as tf
from math import ceil
from keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as keras

import numpy as np
import os

from tifffile import imsave
import glob
import pathlib

from UNet_utils import *

In [2]:
%load_ext watermark

%watermark
%watermark --iversions

Last updated: 2025-04-04T12:18:47.378293+02:00

Python implementation: CPython
Python version       : 3.9.20
IPython version      : 8.18.1

Compiler    : GCC 11.2.0
OS          : Linux
Release     : 5.15.167.4-microsoft-standard-WSL2
Machine     : x86_64
Processor   : x86_64
CPU cores   : 16
Architecture: 64bit

keras     : 3.6.0
tqdm      : 4.66.5
numpy     : 1.26.4
tensorflow: 2.17.0
PIL       : 10.4.0
tifffile  : 2024.8.30
skimage   : 0.24.0



## 2) Check GPU access

In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.17.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1743761931.007781    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1743761931.293027    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1743761931.293075    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


## 3) Load U-Net model

In [11]:
model_path = r''
unet = load_model(model_path, custom_objects={'_weighted_binary_crossentropy': weighted_binary_crossentropy(np.ones(2))})
input_size = unet.input_shape[1:3]

I0000 00:00:1743762201.906227    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1743762201.906301    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1743762201.906319    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1743762203.294781    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1743762203.294860    3949 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-04

Model input size: 320x320


## 4) Run predictions 

In [14]:
data_folder = r'/source'
mask_folder = r'/target'
paths = glob.glob(data_folder + '/*.tif')

import time 
predictions = []
for path in paths:
  s = time.time()
  mask = predict_as_tiles(path, unet)
  mask_name = pathlib.PurePath(path).name.replace('.tif','.png')
  
  save_path = os.path.join(mask_folder, mask_name)
  Image.fromarray(mask).save(save_path, "PNG")
    
  predictions.append(mask)
  e = time.time()
  print(e-s)

I0000 00:00:1743762399.299759    9955 service.cc:146] XLA service 0x7fc738003790 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743762399.299800    9955 service.cc:154]   StreamExecutor device (0): NVIDIA RTX A2000 Laptop GPU, Compute Capability 8.6
2025-04-04 12:26:39.338631: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-04 12:26:39.459572: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


I0000 00:00:1743762403.106248    9955 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━